# Monte Carlo GBM Quant Report
Notebook de ejecucion del motor. La logica principal vive en `monte_carlo_simulatio_proyect.py`.

In [59]:
import logging

import pandas as pd
import seaborn as sns

from monte_carlo_simulatio_proyect import (
    EWMA_LAMBDA,
    NUM_DAYS_MULTIPLE_TICKERS,
    NUM_DAYS_SINGLE_TICKER,
    NUM_SIMULATIONS,
    REPORT_PATH,
    ROLLING_WINDOW,
    SINGLE_TICKER,
    START_DATE,
    STUDENT_T_DF,
    TICKERS,
    VAR_LEVEL,
    create_pdf_report,
    get_close_series,
    get_stock_data,
    run_multiple_tickers,
    setup_logging,
    simulate_dynamic_gbm,
    summarize_results,
)

setup_logging(logging.INFO)
sns.set_theme(style='darkgrid')

## Configuracion

In [60]:
config_df = pd.DataFrame([
    {'Variable': 'START_DATE', 'Valor': START_DATE, 'Uso': 'Inicio de la data historica'},
    {'Variable': 'SINGLE_TICKER', 'Valor': SINGLE_TICKER, 'Uso': 'Accion para prueba individual'},
    {'Variable': 'TICKERS', 'Valor': ', '.join(TICKERS), 'Uso': 'Universo comparativo'},
    {'Variable': 'NUM_SIMULATIONS', 'Valor': NUM_SIMULATIONS, 'Uso': 'Caminos simulados'},
    {'Variable': 'NUM_DAYS_SINGLE_TICKER', 'Valor': NUM_DAYS_SINGLE_TICKER, 'Uso': 'Horizonte individual'},
    {'Variable': 'NUM_DAYS_MULTIPLE_TICKERS', 'Valor': NUM_DAYS_MULTIPLE_TICKERS, 'Uso': 'Horizonte comparativo'},
    {'Variable': 'ROLLING_WINDOW', 'Valor': ROLLING_WINDOW, 'Uso': 'Ventana de varianza reciente'},
    {'Variable': 'EWMA_LAMBDA', 'Valor': EWMA_LAMBDA, 'Uso': 'Peso de volatilidad reciente'},
    {'Variable': 'STUDENT_T_DF', 'Valor': STUDENT_T_DF, 'Uso': 'Colas pesadas'},
    {'Variable': 'VAR_LEVEL', 'Valor': VAR_LEVEL, 'Uso': 'Nivel de VaR/CVaR'},
    {'Variable': 'REPORT_PATH', 'Valor': str(REPORT_PATH), 'Uso': 'Salida PDF'},
])
config_df

,Variable,Valor,Uso
0,START_DATE,1950-01-01,Inicio de la data historica
1,SINGLE_TICKER,AAPL,Accion para prueba individual
2,TICKERS,"AAPL, MSFT, GOOGL, AMZN, META, PLTR",Universo comparativo
3,NUM_SIMULATIONS,10000,Caminos simulados
4,NUM_DAYS_SINGLE_TICKER,252,Horizonte individual
5,NUM_DAYS_MULTIPLE_TICKERS,22,Horizonte comparativo
6,ROLLING_WINDOW,252,Ventana de varianza reciente
7,EWMA_LAMBDA,0.94,Peso de volatilidad reciente
8,STUDENT_T_DF,5,Colas pesadas
9,VAR_LEVEL,0.05,Nivel de VaR/CVaR


## Una Accion

In [61]:
single_data = get_stock_data(SINGLE_TICKER, START_DATE)
single_close = get_close_series(single_data, SINGLE_TICKER)
single_result = simulate_dynamic_gbm(
    single_close,
    SINGLE_TICKER,
    NUM_DAYS_SINGLE_TICKER,
    NUM_SIMULATIONS,
)

pd.DataFrame([{
    'Ticker': single_result.ticker,
    'Precio actual': single_result.current_price,
    'Mediana simulada': single_result.median_final_price,
    'Piso 95%': single_result.lower_bound,
    'Techo 95%': single_result.upper_bound,
    'VaR 5%': single_result.var_5_return * 100,
    'CVaR 5%': single_result.cvar_5_return * 100,
    'Sigma usada diaria': single_result.sigma_used,
    'Error calibracion': single_result.calibration_error,
}]).round(4)

2026-06-24 12:36:21,488 - INFO - Downloading price data | tickers=AAPL | start=1950-01-01
2026-06-24 12:36:23,022 - INFO - Downloaded price data | rows=11474 | columns=5
2026-06-24 12:36:23,029 - INFO - AAPL | Starting simulation | days=252 | simulations=10000
2026-06-24 12:36:23,034 - INFO - AAPL | Log returns ready | observations=11473
2026-06-24 12:36:23,058 - INFO - AAPL | Fetching option chain for implied volatility
2026-06-24 12:36:24,068 - INFO - AAPL | Selected option expiration=2027-02-19
2026-06-24 12:36:24,362 - INFO - AAPL | Daily implied volatility=0.017152
2026-06-24 12:36:24,364 - INFO - AAPL | Parameters | current=296.5900 | mu=0.00096153 | sigma_ewma=0.013964 | sigma_used=0.015558
2026-06-24 12:36:42,290 - INFO - AAPL | Simulation progress 25%
2026-06-24 12:36:59,764 - INFO - AAPL | Simulation progress 50%
2026-06-24 12:37:17,452 - INFO - AAPL | Simulation progress 75%
2026-06-24 12:37:34,747 - INFO - AAPL | Simulation complete | median=367.5608 | VaR5=-16.38% | CVaR5=

,Ticker,Precio actual,Mediana simulada,Piso 95%,Techo 95%,VaR 5%,CVaR 5%,Sigma usada diaria,Error calibracion
0,AAPL,296.59,367.5608,222.2253,583.4411,-16.3827,-28.9515,0.0156,0.0658


## Varias Acciones + PDF

In [62]:
market_data = get_stock_data(TICKERS, START_DATE)
results = []

for ticker in TICKERS:
    close_prices = get_close_series(market_data, ticker)
    result = simulate_dynamic_gbm(
        close_prices,
        ticker,
        NUM_DAYS_MULTIPLE_TICKERS,
        NUM_SIMULATIONS,
    )
    results.append(result)

summary_df = summarize_results(results)
report_path = create_pdf_report(results, summary_df)
print(f'PDF generado: {report_path}')
summary_df.round(4)

2026-06-24 12:37:34,780 - INFO - Downloading price data | tickers=['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'PLTR'] | start=1950-01-01
2026-06-24 12:37:37,997 - INFO - Downloaded price data | rows=11474 | columns=30
2026-06-24 12:37:38,008 - INFO - AAPL | Starting simulation | days=22 | simulations=10000
2026-06-24 12:37:38,011 - INFO - AAPL | Log returns ready | observations=11473
2026-06-24 12:37:38,026 - INFO - AAPL | Fetching option chain for implied volatility
2026-06-24 12:37:39,026 - INFO - AAPL | Selected option expiration=2026-07-17
2026-06-24 12:37:39,219 - INFO - AAPL | Daily implied volatility=0.014717
2026-06-24 12:37:39,221 - INFO - AAPL | Parameters | current=296.5800 | mu=0.00096035 | sigma_ewma=0.013963 | sigma_used=0.014340
2026-06-24 12:37:40,824 - INFO - AAPL | Simulation progress 25%
2026-06-24 12:37:42,431 - INFO - AAPL | Simulation progress 50%
2026-06-24 12:37:43,940 - INFO - AAPL | Simulation progress 75%
2026-06-24 12:37:45,605 - INFO - AAPL | Simulation compl

PDF generado: outputs/monte_carlo_quant_report.pdf


,Ticker,Precio actual,Mediana simulada,Piso 95%,Techo 95%,Cambio % vs actual,VaR 5%,CVaR 5%,Sigma usada diaria,Error calibracion
0,AAPL,296.58,301.9336,264.5013,346.2787,1.8051,-8.5633,-11.9724,0.0143,0.0658
1,AMZN,239.13,234.0862,192.9064,285.7454,-2.1093,-16.2411,-20.9264,0.0213,0.0581
2,GOOGL,348.60,331.2844,273.6507,399.3206,-4.9672,-18.3846,-23.0077,0.0207,0.0745
3,META,556.12,513.9727,410.6406,639.6609,-7.5788,-22.7371,-27.6176,0.0237,0.0729
4,MSFT,371.26,338.2662,278.2142,407.1256,-8.8870,-22.1611,-26.5777,0.0209,0.0843
5,PLTR,112.69,99.2061,71.6078,135.0539,-11.9655,-31.9677,-38.0997,0.0344,0.0636


## Lectura De Riesgo

In [63]:
risk_view = summary_df[[
    'Ticker',
    'Cambio % vs actual',
    'VaR 5%',
    'CVaR 5%',
    'Error calibracion',
]].copy()

risk_view = risk_view.sort_values('CVaR 5%')
risk_view.round(4)

,Ticker,Cambio % vs actual,VaR 5%,CVaR 5%,Error calibracion
5,PLTR,-11.9655,-31.9677,-38.0997,0.0636
3,META,-7.5788,-22.7371,-27.6176,0.0729
4,MSFT,-8.8870,-22.1611,-26.5777,0.0843
2,GOOGL,-4.9672,-18.3846,-23.0077,0.0745
1,AMZN,-2.1093,-16.2411,-20.9264,0.0581
0,AAPL,1.8051,-8.5633,-11.9724,0.0658
